[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/10_Superficie_Agricola/01_superficie_agricola.ipynb)

# Superficie agricola en el tiempo: serie, frecuencia y cambio
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

# M11. Superficie agrícola en el tiempo (30 años): serie, frecuencia y cambio
**Dimensiones:** *tendencia* (km² por década), *frecuencia* (% de años con cultivo en el píxel),
*ocurrencia* (superficie actual), *cambio* (ganancia/pérdida entre primera y última observación).

> **📌 Regla de los 30 años:** la ventana CONFIG es **1996–2026**. El producto anual de
> coberturas `MODIS/061/MCD12Q1` inicia en **2001**: los años **1996–2000 se muestran como
> "sin dato"** (se declara, no se inventa). Si se necesita cubrirlos, ver alternativas abajo.

* **Primario:** `MODIS/061/MCD12Q1` — clasificación anual `LC_Type1` (500 m), clases
  **12 = cultivos** y **14 = mosaico cultivos/vegetación natural** (configurables en
  `UMBRALES["clases_agricolas"]`), disponible **2001–actual**.
* **Alternativas (comentar primaria y descomentar):**
  * `GOOGLE/DYNAMICWORLD` — clase `crops`, **10 m, 2015+**: detalle intra-parcela;
  * `ESA/WorldCover/v200` clase 40 — **10 m, 2020/2021**: instantánea de alta resolución (M12);
  * **Antes de 2001:** no existe producto global anual; requeriría una **clasificación propia
    con Landsat** (5/7/8) — fuera del alcance de esta plantilla, se documenta como pendiente.

<details>
<summary><b>📐 Qué mide cada salida de este módulo</b> — clic para ver</summary>
<br>

| Salida | Dimensión | Pregunta que responde |
|---|---|---|
| Serie anual km² (M11.1) | tendencia | ¿se gana o se pierde tierra cultivada en 30 años? |
| % del AOI por año | ocurrencia | ¿qué proporción del territorio se cultiva? |
| Mapa de frecuencia (M11.2) | frecuencia | ¿en qué píxeles se cultiva casi siempre vs. a veces? |
| Mapa primero→último (M11.2) | cambio | ¿dónde hubo ganancia y dónde pérdida de superficie agrícola? |

**Límites a declarar:** resolución 500 m (mezcla píxeles de borde), clase 14 incluye mosaico con
vegetación natural (puede sobreestimar), años sin dato antes de 2001, y el último año del
producto puede tener rezago de publicación.

</details>

In [ ]:
# ============================================================
# M11.1 Serie de superficie agrícola por año (1996-2026 -> dato 2001+)
# ============================================================

# --- Primario: MCD12Q1 anual (2001+) ------------------------------------------
lc = (
    ee.ImageCollection("MODIS/061/MCD12Q1")
    .filterDate(FECHA_INICIO, FECHA_FIN)
    .filterBounds(roi)
)

# --- Alternativas (comentar primaria y descomentar) ---------------------------
# Dynamic World 10 m (2015+), probabilidad de cultivos > 0.5:
# lc = (
#     ee.ImageCollection("GOOGLE/DYNAMICWORLD")
#     .filterDate("2019-01-01", FECHA_FIN).filterBounds(roi)
#     .select("crops").map(lambda i: i.gt(0.5).rename("LC_Type1"))
# )
# WorldCover 10 m (2020/2021), recodificar cultivos (40) a 12:
# lc = (
#     ee.ImageCollection("ESA/WorldCover/v200")
#     .filterDate("2020-01-01", "2022-01-01")
#     .map(lambda i: i.select("Map").remap([40], [12], 0).rename("LC_Type1"))
# )

# Años realmente disponibles (se declara el rezago, no se rellena)
_fechas_l = pd.to_datetime(lc.aggregate_array("system:time_start").getInfo(), unit="ms")
anios_lc = sorted({int(a) for a in _fechas_l.year.unique()})
anios_agr = [y for y in range(max(ANIO_INI, anios_lc[0]), ANIOS_COMPLETOS[-1] + 1)
             if y in set(anios_lc)]
sin_dato = [y for y in range(ANIO_INI, ANIOS_COMPLETOS[-1] + 1) if y not in set(anios_lc)]

print(f"Ventana CONFIG: {ANIO_INI}-{ANIOS_COMPLETOS[-1]} (regla 30 años)")
print(f"Años con dato de cobertura agrícola: {anios_agr[0]}-{anios_agr[-1]} "
      f"({len(anios_agr)} años)")
if sin_dato:
    print(f"Años SIN dato (se excluyen, no se estiman): {sin_dato[0]}-{sin_dato[-1]}")

# --- Máscara de cultivos (clases configurables en CONFIG) ---------------------
CLASES_AGR = UMBRALES["clases_agricolas"]


def _mascara_agr(img):
    m = img.eq(CLASES_AGR[0])
    for c in CLASES_AGR[1:]:
        m = m.Or(img.eq(c))
    return m


mascaras_agr, imgs_agr = [], []
for y in anios_agr:
    img_y = lc.filter(ee.Filter.calendarRange(y, y, "year")).first().select("LC_Type1")
    m = _mascara_agr(img_y)
    mascaras_agr.append(m)
    imgs_agr.append(pix_km2.updateMask(m).rename(f"a_{y}"))

d_agr = reducir_bandas(
    imgs_agr, [f"a_{y}" for y in anios_agr], ESCALAS["modis"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
df_agr = pd.DataFrame({
    "anio": anios_agr,
    "km2_agricola": [d_agr.get(f"a_{y}") or 0 for y in anios_agr],
})
df_agr["pct_zona"] = 100 * df_agr["km2_agricola"] / area_km2

# --- Métricas de tendencia -----------------------------------------------------
n_ref = min(3, len(df_agr))
prim_ref = df_agr["km2_agricola"].head(n_ref).mean()
ult_ref = df_agr["km2_agricola"].tail(n_ref).mean()
agr_delta_km2 = ult_ref - prim_ref
agr_delta_pct = 100 * agr_delta_km2 / max(prim_ref, 1e-9)
pend_agr = np.polyfit(df_agr["anio"], df_agr["km2_agricola"], 1)[0] * 10

print(f"Superficie agrícola promedio (últimos {n_ref} años): {ult_ref:,.1f} km² "
      f"({100 * ult_ref / area_km2:.1f}% del AOI)")
print(f"Cambio {anios_agr[0]}-> {anios_agr[-1]} "
      f"(prom. {n_ref} años): {agr_delta_km2:+,.1f} km² ({agr_delta_pct:+.1f}%)")
print(f"Tendencia: {pend_agr:+.1f} km² por década")
print(f"Año con mayor superficie: {int(df_agr.loc[df_agr['km2_agricola'].idxmax(), 'anio'])} | "
      f"menor: {int(df_agr.loc[df_agr['km2_agricola'].idxmin(), 'anio'])}")

display(df_agr.round(1))

# --- Gráfico con marca de años sin dato ----------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.8))
if sin_dato:
    ax.axvspan(ANIO_INI - 0.5, anios_agr[0] - 0.5, color="0.9", label="sin dato (pre-2001)")
ax.plot(df_agr["anio"], df_agr["km2_agricola"], marker="o", ms=4,
        lw=1.6, color="#1b8a3c", label="superficie agrícola")
ax.plot(df_agr["anio"],
        np.polyval(np.polyfit(df_agr["anio"], df_agr["km2_agricola"], 1), df_agr["anio"]),
        color="#ff9800", lw=2, label=f"tendencia {pend_agr:+.1f} km²/década")
ax.set_title("Superficie agrícola por año (MCD12Q1, clases 12+14) — ventana de 30 años")
ax.set_ylabel("km² con cultivo")
ax.set_xlabel("Año")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

res_m11 = {
    "agr_km2": float(df_agr["km2_agricola"].iloc[-1]),
    "agr_pct": float(df_agr["pct_zona"].iloc[-1]),
    "agr_anio_ref": int(df_agr["anio"].iloc[-1]),
    "agr_delta_km2": float(agr_delta_km2),
    "agr_delta_pct": float(agr_delta_pct),
    "agr_pend_decada": float(pend_agr),
    "agr_prim_ref": float(prim_ref),
}

In [ ]:
# ============================================================
# M11.2 Mapas: frecuencia de cultivo y cambio (primero -> último año)
# ============================================================

# --- 1) Frecuencia: % de años con cultivo en cada píxel ------------------------
frec_pct = (
    ee.ImageCollection(mascaras_agr)
    .reduce(ee.Reducer.sum())
    .divide(len(mascaras_agr))
    .multiply(100)
    .rename("pct_anios_cultivo")
)

frec_aoi = promediar_aoi(frec_pct, "pct_anios_cultivo", ESCALAS["modis"])
print(f"Frecuencia media de cultivo en el AOI: {frec_aoi:.1f}% de los años "
      f"({anios_agr[0]}-{anios_agr[-1]})")

# --- 2) Cambio: primero vs. último año con dato --------------------------------
ag_prim = mascaras_agr[0]
ag_ult = mascaras_agr[-1]

# 0 = nunca cultivado | 1 = ganancia (solo al último) | 2 = pérdida (solo al primero)
# 3 = estable (siempre cultivado)
trans = ag_prim.multiply(2).add(ag_ult).rename("transicion")

gana = pix_km2.updateMask(ag_ult.And(ag_prim.Not())).rename("gana")
pierde = pix_km2.updateMask(ag_prim.And(ag_ult.Not())).rename("pierde")
estable = pix_km2.updateMask(ag_prim.And(ag_ult)).rename("estable")
d_trans = reducir_bandas(
    [gana, pierde, estable], ["gana", "pierde", "estable"], ESCALAS["modis"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
gan_km2 = d_trans.get("gana") or 0
perd_km2 = d_trans.get("pierde") or 0
est_km2 = d_trans.get("estable") or 0

print(f"Cambio {anios_agr[0]} -> {anios_agr[-1]}: "
      f"ganancia {gan_km2:,.1f} km² | pérdida {perd_km2:,.1f} km² | "
      f"estable {est_km2:,.1f} km²")

res_m11.update({"agr_gan_km2": float(gan_km2), "agr_perd_km2": float(perd_km2)})

# --- 3) Mapas -------------------------------------------------------------------
vis_freq = {"min": 0, "max": 100,
            "palette": ["f7fcf5", "c7e9c0", "74c476", "238b45", "00441b"]}
vis_trans = {"min": 0, "max": 3,
             "palette": ["f2f2f2", "1b8a3c", "d73027", "2166ac"]}

Map_m11 = geemap.Map()
Map_m11.centerObject(roi, ZONA["zoom"])
Map_m11.add_basemap("HYBRID")
Map_m11.addLayer(frec_pct, vis_freq,
                 f"Frecuencia de cultivo (% de {len(mascaras_agr)} años)")
Map_m11.addLayer(trans, vis_trans, "Cambio primero->último año", False)
Map_m11.addLayer(borde_aoi(), {}, "AOI")
Map_m11.add_colorbar(vis_freq, label="% de años con cultivo",
                     orientation="horizontal", position="bottomright",
                     layer_name="Frecuencia de cultivo (%)")
Map_m11

## Interpretación de M11
* **Serie (M11.1):** pendiente positiva = expansión de la frontera agrícola; negativa = abandono,
  conversión a otra cobertura o presión urbana/minera. El **% del AOI** dimensiona el peso del
  sector agrícola en el territorio.
* **Frecuencia (mapa verde):** > 80% = agricultura de temporada estable; 20–60% = cultivos
  alternados con barbecho o mosaico — zonas de decisión más incierta para los programas.
* **Cambio (mapa):** 🔵 **azul = estable**, 🟢 **verde = ganancia** de superficie agrícola,
  🔴 **rojo = pérdida**, ⚪ **gris = nunca cultivado**. Cruzar pérdidas con M12 (¿se volvió
  urbano? ¿mina?) y con M8 (¿se secó el agua?).
* **Comparación de resolución:** M11 da la *evolución* (500 m, 25 años); M12 da el *estado actual*
  (10 m). Esperar diferencias de magnitud en bordes de parcela.

**Preguntas guía**
1. ¿La expansión o pérdida de superficie agrícola coincide con lo que las comunidades relatan
   (colonización, abandono, minería, urbanización)?
2. ¿Dónde está la "frecuencia intermedia" (zonas de rotación)? ¿son barbechos o incertidumbre?
3. ¿Los años de caída de la serie agrícola son años de sequía severa (M5) y bajo NDVI (M7)?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
